# SWE-chat Data Check: Following the Proposal's Data-Processing Requirements

This notebook checks SWE-chat against the *Task and data* section of the proposal:

1. Keep only repositories with **1 to 2 contributors** and **at least k+1 sessions** (k ∈ {1, 2, 4, 8}); order each repository's sessions by start time.
2. Targets are the **agent turns** in sessions after the first k. The label is `prompt_pushback` of the **user prompt that follows**: T1 is binary, T2 is four-way.
3. Split **by repository** into train/holdout/test = 70/15/15, so that **no developer appears in more than one split**.

**Setup**: use the `.venv-swechat` environment (kernel `Python (swechat)`) from the project directory. The data is already downloaded to `data/raw/`; `scripts/load_data.py` reads the local file and caches the columns we need.

**Changes from the previous version**
- Data is read from local files instead of remotely.
- **Stricter definition of a target turn** (Section 3): non-human prompts are removed (environment injections, slash commands, interruption markers, automated templates, etc.); there must be **substantive agent activity** between two human prompts (at least one tool call, or one non-empty, non-error text reply); service-failure replies such as `API Error` do not count.
- **The split** now merges repositories that share a user into one group before splitting, which prevents developer leakage.

## First, what is k?

**k = the number of history sessions the agent may look at**, i.e. how many early sessions are used to build a developer's (repository's) preference profile.

For each repository, sort its sessions by start time: `s1, s2, s3, ...`. Once k is fixed:

- **History (read-only)**: the first k sessions `s1 … sk`. The predictor may only use these to learn the developer's preferences, for example which behaviors were corrected in the past.
- **Targets (to predict)**: every agent turn in session k+1 and later. For each turn we predict whether the user will push back next.
- Target sessions must not be used to build the profile and must not "peek into the future".

Example: a repository has 6 sessions and k=2.

```
s1  s2  |  s3  s4  s5  s6
history (2) |  target sessions: predict the reaction to every agent turn in these 4 sessions
```

**The proposal uses k ∈ {1, 2, 4, 8} to draw a learning curve**: does more history make prediction better?

**How k affects the amount of data (used in the tables below)**
- A repository needs at least **k+1 sessions** to qualify (k for history, at least 1 for targets). So the larger k is, the fewer repositories qualify.
- The larger k is, the fewer sessions remain as targets per repository, but the history is richer.
- For example, the k=8 repository set is a subset of the k=1 set. For the train/holdout/test split we split the k=1 set once and reuse that split for larger k, so the split does not change with k.

**Relation to the baseline**: the B1 baseline is "the pushback rate over the repository's first k sessions", so B1 also depends on k.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "scripts")
from load_data import load
from targets import build_targets, NON_HUMAN_PREFIXES

pd.set_option("display.width", 140)
SEED = 0
KS = [1, 2, 4, 8]
LABELS = ["correction", "rejection", "failure_report", "non_pushback"]

## 1. Load the data and basic checks

The README reports 707 repositories, 17,968 sessions and 229,909 user prompts.

`turns` is a compact table extracted from the `conversations` table (user prompts, agent text replies and tool calls; text is truncated to the first 300 characters). It is produced by `scripts/build_turns.py`.

In [2]:
sessions, repos, _ = load()
turns = pd.read_parquet("data/turns_compact.parquet")
sessions["start"] = pd.to_datetime(sessions.session_started_at, utc=True, errors="coerce").fillna(
    pd.to_datetime(sessions.created_at, utc=True, errors="coerce"))

print(f"repositories: {len(repos)}   sessions: {len(sessions):,}   user prompts: {(turns.turn_type == 'user_prompt').sum():,}")
print(f"repositories seen in sessions: {sessions.repo_id.nunique()}   duplicate session_ids: {sessions.session_id.duplicated().sum()}")
print(f"sessions missing a start time: {sessions.start.isna().sum()}   sessions missing user_id: {sessions.user_id.isna().sum():,}")
print(f"time range: {sessions.start.min()} -> {sessions.start.max()}")
sessions.agent.value_counts().head(6)

repositories: 707   sessions: 17,968   user prompts: 229,909
repositories seen in sessions: 705   duplicate session_ids: 0
sessions missing a start time: 2   sessions missing user_id: 5,591
time range: 2026-01-05 14:18:08.586085+00:00 -> 2026-09-04 15:20:51.045885+00:00


agent
Claude Code       12417
Codex              2362
OpenCode           1525
Project Runner      396
Pi                  368
Cursor              318
Name: count, dtype: int64

## 2. Definition of "contributor": do the two counts agree?

- `repositories.num_contributors_in_dataset`: the number of distinct commit authors for the repository in the dataset.
- The number of distinct `user_id` values per repository in the sessions table.

If the two agree, either one can be used for filtering.

In [3]:
sc = sessions.groupby("repo_id").agg(n_sessions=("session_id", "size"), n_users=("user_id", "nunique"))
rp = repos.set_index("repo_id").join(sc, how="outer")
print(pd.crosstab(rp.num_contributors_in_dataset.clip(upper=4), rp.n_users.clip(upper=4), dropna=False,
                  rownames=["contributors per repositories table (4 = 4+)"], colnames=["distinct user_ids in sessions"]))
print("repositories where the two counts disagree:", (rp.num_contributors_in_dataset != rp.n_users).sum())

distinct user_ids in sessions                 0.0  1.0  2.0  3.0  4.0  NaN
contributors per repositories table (4 = 4+)                              
0                                              44    0    0    0    0    0
1                                               1  605    0    0    0    2
2                                               0    0   39    0    0    0
3                                               0    0    0   10    0    0
4                                               0    0    0    0    6    0
repositories where the two counts disagree: 3


## First, what is a "turn", how is it counted, and why use it?

**A turn = one round of "agent works → user responds".** The user sends a prompt, the agent does some work (text replies, tool calls that edit code, etc.), and the user then sends the **next prompt**. What we predict is the reaction type of that next prompt.

```
human prompt #1 ──► agent works (replies, edits code, runs commands…) ──► human prompt #2   ← the label prompt_pushback sits here
                    └──────────── one turn (the object we predict) ────────────┘
human prompt #2 ──► agent works ──► human prompt #3   ← the next turn
```

### How it is counted (matches the code in Section 3)
A user prompt becomes a **target turn** only if all of the following hold:
1. **It is labeled**: `prompt_pushback` is not null.
2. **It was written by a human**: not an environment injection, slash command, `[Request interrupted…]`, Codex approval template, or other automatically generated content.
3. **It has an earlier human prompt**: i.e. it is not the first human prompt in the session (nothing precedes the first one for it to react to).
4. **The agent did substantive work between the two human prompts**: at least one tool call, or at least one non-empty, non-error text reply. Service failures and placeholders such as `API Error: 529 Overloaded`, `Prompt is too long`, `You've hit your … limit` and `No response requested.` do not count.

Each prompt that qualifies is one turn, and its label is its `prompt_pushback`:
- T1: `non_pushback` is 0; `correction / rejection / failure_report` is 1.
- T2: keep the four classes.

Then, following Section 4, keep only repositories with 1 to 2 contributors and at least k+1 sessions, and count only session k+1 and later.

Note: **the number of turns is neither the number of sessions nor the number of agent replies.** One session can contain dozens of turns.

### Why turns rather than sessions
- **The proposal's task is to predict the user's reaction to a new agent turn.** What the agent should avoid later is "this edit gets rejected"; the granularity is a single edit, not a whole session.
- **The labels are turn-level by nature**: SWE-chat's `prompt_pushback` sits on every user prompt, and there is no session-level label. Aggregating to sessions would lose "which edit went wrong", and longer sessions are more likely to contain pushback, which creates a length confound.
- **Far more samples**: about 13 thousand target sessions correspond to roughly 70 to 80 thousand turns.
- **The reward granularity matches**: later we use the "no-pushback probability" as a reward, which scores a single agent reply and needs turn-level predictions.

### Two things to keep in mind
- **Pushback across turns is not independent**: turns in the same session or repository are highly correlated, so evaluation must bootstrap over repositories and must not treat all turns as independent samples.
- **The distribution of turns is very uneven**: repositories differ a lot in how many turns they have (see Section 7), so watch out for a few large repositories dominating aggregate metrics.

## 3. Label distribution and construction of target turns

`prompt_pushback` sits on a user prompt and describes that prompt's reaction to the **previous stretch of agent output**. The rules are in the previous section and the implementation is in `scripts/targets.py`; the funnel below shows how many prompts each step removes.

Prefixes used to exclude "non-human prompts" (a match on any of them excludes the prompt; prompts that start with a `/command-name` slash command are excluded as well):

In [4]:
print("Label distribution over all user prompts:")
print(turns[turns.turn_type == "user_prompt"].prompt_pushback.value_counts(dropna=False).to_string())

tg, funnel = build_targets(turns)
funnel["share_of_prev"] = (funnel.prompts / funnel.prompts.shift()).round(3)
print("\nTarget-turn funnel:")
print(funnel.to_string(index=False))
print("\nNon-human prompt prefixes:", [p for p in NON_HUMAN_PREFIXES])

assert tg.prompt_pushback.isin(LABELS).all()
print(f"\nTarget turns: {len(tg):,}   T1 pushback rate: {tg.t1.mean():.3f}   (proposal cites 51%)")
print("T2 distribution:", tg.prompt_pushback.value_counts(normalize=True).round(3).to_dict())

Label distribution over all user prompts:
prompt_pushback
non_pushback      121742
correction         63710
NaN                28424
failure_report     12888
rejection           3145



Target-turn funnel:
                                                step  prompts  share_of_prev
                                    all user prompts   229909            NaN
                               with a pushback label   201485          0.876
                ... after removing non-human prompts   175814          0.873
... after removing each session's first human prompt   159589          0.908
      ... after requiring substantive agent activity   111034          0.696

Non-human prompt prefixes: ['[Request interrupted', '<environment_context', '<command-', '<local-command', '<system-reminder', '<task-notification', '<subagent_notification', '<turn_aborted', '<codex_internal_context', '<recommended_plugins', '# AGENTS.md instructions', '>>> ', 'The following is the Codex agent history', 'The Codex agent has re', 'Reviewed Codex session', 'Assess the exact plann', 'Planned action JSON', 'Another Claude session', '[delegate worktree]', 'You are a code reviewer', '## Output', '##

### Spot check: composition of the agent's activity

- `agent_has_text`: whether the agent produced a text reply between the two human prompts. Turns with tool calls only (no text) are rare, and their pushback rate is close to that of turns with text, so under the new definition the two groups no longer differ noticeably.
- `n_tool` / `n_text`: the number of tool calls / valid text replies.

In [5]:
print(tg.groupby("agent_has_text").agg(turns=("t1", "size"), t1_rate=("t1", "mean")).round(3))
print(f"Turns with text only (no tool calls): {(tg.n_tool == 0).sum():,}")
print(f"Turns that also contained API Error-style failure replies: {(tg.n_bad_text > 0).sum():,}")
print("\nMost common beginnings of the remaining prompts (manual check for missed automated templates):")
tg["head"].fillna("").str.lstrip().str.slice(0, 28).value_counts().head(15).to_frame("n")

                 turns  t1_rate
agent_has_text                 
False             1411    0.524
True            109623    0.533
Turns with text only (no tool calls): 23,749
Turns that also contained API Error-style failure replies: 5,913

Most common beginnings of the remaining prompts (manual check for missed automated templates):


,n
head,
yes,1785
update,692
commit,652
continue,556
Yes,453
commit and push,415
go,312
Continue from where you left,308
1,284


## 4. Repository filter: 1 to 2 contributors and at least k+1 sessions

Sort each repository's sessions by start time; the first k are history, and the agent turns in later sessions are targets.

Each row of the table below corresponds to one k (see "What is k?" at the top):
- **repos**: the number of repositories with at least k+1 sessions.
- **history_sessions**: the total number of first-k sessions across those repositories, used only to build the preference profile.
- **target_sessions / target_turns**: the number of sessions from k+1 onwards and the number of agent turns to predict in them.
- **T1_pushback_rate**: the fraction of target turns with pushback.
- **repos_with_targets**: repositories that have at least one target turn.

In [6]:
ok = sessions.merge(repos[["repo_id", "num_contributors_in_dataset"]], on="repo_id", how="left")
ok = ok[ok.num_contributors_in_dataset.between(1, 2)].sort_values(["repo_id", "start", "session_id"]).copy()
ok["rank"] = ok.groupby("repo_id").cumcount()
ok["n_sess"] = ok.groupby("repo_id").session_id.transform("size")
print(f"Repositories with 1-2 contributors: {ok.repo_id.nunique()} / {sessions.repo_id.nunique()}, sessions {len(ok):,}")

tg = tg.merge(ok[["session_id", "repo_id", "user_id", "rank", "n_sess", "start"]], on="session_id", how="inner")
print(f"Target turns in these repositories: {len(tg):,}")

rows = []
for k in KS:
    d = ok[ok.n_sess >= k + 1]
    t = tg[(tg.n_sess >= k + 1) & (tg["rank"] >= k)]
    rows.append({"k": k, "repos": d.repo_id.nunique(), "history_sessions": int((d["rank"] < k).sum()),
                 "target_sessions": int((d["rank"] >= k).sum()), "target_turns": len(t),
                 "T1_pushback_rate": round(t.t1.mean(), 3), "repos_with_targets": t.repo_id.nunique()})
pd.DataFrame(rows)

Repositories with 1-2 contributors: 645 / 705, sessions 14,251
Target turns in these repositories: 81,978


,k,repos,history_sessions,target_sessions,target_turns,T1_pushback_rate,repos_with_targets
0,1,519,519,13606,74532,0.530,500
1,2,448,896,13087,70455,0.530,437
2,4,375,1500,12229,63835,0.528,358
3,8,287,2296,10882,54313,0.530,274


### Sample sizes of the four T2 classes for different k

`rejection` is very rare, which directly affects whether the per-class T2 metrics are trustworthy.

In [7]:
rows = []
for k in KS:
    t = tg[(tg.n_sess >= k + 1) & (tg["rank"] >= k)]
    rows.append({"k": k, **t.prompt_pushback.value_counts().reindex(LABELS).to_dict()})
pd.DataFrame(rows).set_index("k")

,correction,rejection,failure_report,non_pushback
k,,,,
1,31796,1514,6223,34999
2,30067,1432,5874,33082
4,27088,1305,5325,30117
8,23117,1121,4569,25506


## 5. Time-ordering checks

In [8]:
print("Sessions with tied start times within a repository:", ok.duplicated(["repo_id", "start"]).sum())
print("Sessions missing a start time:", ok.start.isna().sum())
bad = ok.groupby("repo_id").apply(
    lambda g: g.sort_values("created_at").session_id.tolist() != g.sort_values(["start", "session_id"]).session_id.tolist(),
    include_groups=False)
print("Repositories where created_at order differs from session_started_at order:", int(bad.sum()))

Sessions with tied start times within a repository: 14
Sessions missing a start time: 1


Repositories where created_at order differs from session_started_at order: 2


## 6. Repository-level 70/15/15 split, with developers kept within one split

If we only split repositories at random, a user who has sessions in several repositories can end up in both train and test, which violates the proposal's "no developer appears in more than one split".

Approach: merge repositories that share a `user_id` into one group with union-find, then **assign groups** randomly to the three splits so that repository counts are roughly 70/15/15. The split is based on the k=1 repository set (at least 2 sessions); larger-k sets are subsets of it and reuse the same split.

In [9]:
split_repos = sorted(ok[ok.n_sess >= 2].repo_id.unique())

parent = {r: r for r in split_repos}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

u = ok[ok.repo_id.isin(split_repos) & ok.user_id.notna()].groupby("user_id").repo_id.unique()
for repos_of_user in u:
    for r in repos_of_user[1:]:
        union(repos_of_user[0], r)

groups = pd.Series({r: find(r) for r in split_repos}).rename("group")
gsize = groups.value_counts()
print(f"{len(split_repos)} repositories -> {len(gsize)} connected groups; largest group has {gsize.iloc[0]} repositories; {(gsize > 1).sum()} multi-repository groups")

rng = np.random.default_rng(SEED)
order = rng.permutation(gsize.index.to_numpy())
target = {"train": 0.70, "holdout": 0.15, "test": 0.15}
counts, assign = {s: 0 for s in target}, {}
for g in order:
    # put the group into the split that is currently furthest below its target share
    s = min(target, key=lambda s: counts[s] / (target[s] * len(split_repos)))
    assign[g] = s
    counts[s] += gsize[g]
split = groups.map(assign).rename("split")
print("Repositories per split:", counts)

519 repositories -> 414 connected groups; largest group has 6 repositories; 66 multi-repository groups
Repositories per split: {'train': np.int64(362), 'holdout': np.int64(78), 'test': np.int64(79)}


In [10]:
tg["split"] = tg.repo_id.map(split)
tt = tg[tg.split.notna()]
print(tt.groupby("split").agg(repos=("repo_id", "nunique"), turns=("t1", "size"), t1_rate=("t1", "mean")).round(3))
print()
print(pd.crosstab(tt.split, tt.prompt_pushback, normalize="index").round(3))

# leakage check
su = ok[ok.repo_id.isin(split.index)].assign(split=lambda x: x.repo_id.map(split))
u2s = su.dropna(subset=["user_id"]).groupby("user_id").split.nunique()
print(f"\nUsers appearing in more than one split: {(u2s > 1).sum()} / {len(u2s)}")
print(f"Repositories appearing in more than one split: {split.index.duplicated().sum()}")
assert (u2s > 1).sum() == 0

         repos  turns  t1_rate
split                         
holdout     76  12987    0.516
test        77  10172    0.540
train      357  57469    0.532

prompt_pushback  correction  failure_report  non_pushback  rejection
split                                                               
holdout               0.423           0.072         0.484      0.022
test                  0.419           0.094         0.460      0.027
train                 0.428           0.085         0.468      0.019

Users appearing in more than one split: 0 / 452
Repositories appearing in more than one split: 0


## 7. Skew across repositories

- If sessions or turns concentrate in a few repositories, evaluation should bootstrap over repositories (the proposal's success criterion is exactly a per-repository bootstrap).

In [11]:
per = tt.groupby("repo_id").agg(turns=("t1", "size"), rate=("t1", "mean"))
print(per.turns.describe().round(1).to_string())
print(f"\nRepositories with 0% pushback: {(per.rate == 0).sum()}，with 100% pushback: {(per.rate == 1).sum()}, out of {len(per)}")
print(f"Share of target turns held by the 5 largest repositories: {per.turns.nlargest(5).sum() / per.turns.sum():.3f}")
print("Distribution of per-repository T1 pushback rate:")
per.rate.describe().round(3)

count     510.0
mean      158.1
std       300.2
min         1.0
25%        23.0
50%        66.0
75%       145.5
max      3029.0

Repositories with 0% pushback: 13，with 100% pushback: 4, out of 510
Share of target turns held by the 5 largest repositories: 0.129
Distribution of per-repository T1 pushback rate:


count    510.000
mean       0.516
std        0.185
min        0.000
25%        0.410
50%        0.523
75%        0.647
max        1.000
Name: rate, dtype: float64

## 8. Preliminary check: the repository's own history (B1) vs. a constant baseline (B0)

As defined in the proposal:
- **B0**: the training-set average pushback rate (a constant, so its AUROC is always 0.5; only Brier is compared).
- **B1**: the pushback rate over the repository's first k sessions.

This is only a sanity check on the data (does B1 carry ranking information?), not a formal result. B1 uses the pushback rate over all target turns in the history sessions; repositories with no labeled history turns are skipped.

In [12]:
from sklearn.metrics import roc_auc_score

b0 = tg[tg.split == "train"].t1.mean()
rows = []
for k in KS:
    hist = tg[(tg.n_sess >= k + 1) & (tg["rank"] < k)].groupby("repo_id").t1.mean().rename("b1")
    t = tg[(tg.n_sess >= k + 1) & (tg["rank"] >= k)].join(hist, on="repo_id", how="inner")
    for s in ["holdout", "test"]:
        d = t[t.split == s]
        if d.t1.nunique() < 2:
            continue
        rows.append({"k": k, "split": s, "target_turns": len(d), "repos": d.repo_id.nunique(),
                     "T1 AUROC (B1)": roc_auc_score(d.t1, d.b1),
                     "Brier B0": ((b0 - d.t1) ** 2).mean(), "Brier B1": ((d.b1 - d.t1) ** 2).mean()})
pd.DataFrame(rows).round(3)

,k,split,target_turns,repos,T1 AUROC (B1),Brier B0,Brier B1
0,1,holdout,10681,58,0.515,0.249,0.293
1,1,test,8224,62,0.481,0.249,0.302
2,2,holdout,11288,60,0.489,0.250,0.289
3,2,test,7746,60,0.511,0.249,0.280
4,4,holdout,10569,52,0.563,0.250,0.277
5,4,test,7161,51,0.521,0.249,0.264
6,8,holdout,8547,36,0.596,0.250,0.250
7,8,test,5702,40,0.533,0.249,0.252


## 9. Save the processed results

Later modeling reads these two files directly.

In [13]:
keep = ["turn_id" if "turn_id" in tg else "turn_number", "session_id", "repo_id", "user_id", "turn_number", "rank", "n_sess",
        "start", "agent_has_text", "n_tool", "n_text", "prompt_pushback", "t1", "split"]
keep = list(dict.fromkeys(c for c in keep if c in tg.columns))
tg[keep].to_parquet("data/targets.parquet")
split.rename_axis("repo_id").reset_index().to_parquet("data/repo_split.parquet")
print(f"saved data/targets.parquet ({len(tg):,} rows) and data/repo_split.parquet ({len(split)} repositories)")

saved data/targets.parquet (81,978 rows) and data/repo_split.parquet (519 repositories)
